# Brain size, IQ, and wages — prompt-driven analysis

This notebook was produced by describing the **datasets and scientific questions** to an AI assistant (not by pasting the SciPy lecture code). See [`PROMPTS.md`](PROMPTS.md) for the prompts used.

**Data files** (same folder as this notebook):
- `brain_size.csv` — Willerman et al. (1991): MRI brain volume, height, weight, and IQ scores for 40 adults
- `iris.csv` — Fisher's iris measurements by species
- `wages.txt` — 1985 CPS wage sample (CMU)


## 1. Load and explore the brain-size study

**Question:** What does this table contain, and how do men and women differ on average for verbal IQ and MRI volume?


In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

# Semicolon-separated CSV; "." marks missing weight/height for some subjects
brain = pd.read_csv("brain_size.csv", sep=";", na_values=".")
print(brain.shape)
print(brain.dtypes)
brain.head()


(40, 8)
Unnamed: 0      int64
Gender         object
FSIQ            int64
VIQ             int64
PIQ             int64
Weight        float64
Height        float64
MRI_Count       int64
dtype: object


,Unnamed: 0,Gender,FSIQ,VIQ,PIQ,Weight,Height,MRI_Count
0,1,Female,133,132,124,118.0,64.5,816932
1,2,Male,140,150,124,NaN,72.5,1001121
2,3,Male,139,123,150,143.0,73.3,1038437
3,4,Male,133,129,128,172.0,68.8,965353
4,5,Female,137,132,134,147.0,65.0,951545


In [2]:
# Overall and by-gender summaries for the main outcomes
print("Mean VIQ (all):", brain["VIQ"].mean())
print(brain.groupby("Gender").size())  # sample sizes
print(brain.groupby("Gender")[["VIQ", "FSIQ", "PIQ", "MRI_Count", "Weight", "Height"]].mean())

# Log MRI counts by gender (tutorial exercise)
print("Mean log10(MRI_Count) by gender:")
print(np.log10(brain["MRI_Count"]).groupby(brain["Gender"]).mean())


Mean VIQ (all): 112.35
Gender
Female    20
Male      20
dtype: int64
           VIQ   FSIQ     PIQ  MRI_Count      Weight     Height
Gender                                                         
Female  109.45  111.9  110.45   862654.6  137.200000  65.765000
Male    115.25  115.0  111.60   954855.4  166.444444  71.431579
Mean log10(MRI_Count) by gender:
Gender
Female    5.934994
Male      5.979250
Name: MRI_Count, dtype: float64


## 2. Hypothesis tests on IQ and body measures

**Questions:**
1. Is mean verbal IQ different from zero? (sanity check)
2. Do males and females differ in VIQ?
3. Within person, do full-scale IQ (FSIQ) and performance IQ (PIQ) differ?
4. Do males and females differ in weight? In VIQ under a non-parametric test?


In [3]:
# 1-sample: is mean VIQ = 0? (expect strong rejection)
print("1-sample t-test VIQ vs 0:", stats.ttest_1samp(brain["VIQ"], 0))

female_viq = brain.loc[brain["Gender"] == "Female", "VIQ"]
male_viq = brain.loc[brain["Gender"] == "Male", "VIQ"]
print("2-sample t-test VIQ by gender:", stats.ttest_ind(female_viq, male_viq))

# Paired FSIQ vs PIQ (same individuals)
print("Paired t-test FSIQ vs PIQ:", stats.ttest_rel(brain["FSIQ"], brain["PIQ"]))
print("Wilcoxon signed-rank FSIQ vs PIQ:", stats.wilcoxon(brain["FSIQ"], brain["PIQ"]))

# Weight by gender (drop missing)
female_w = brain.loc[brain["Gender"] == "Female", "Weight"].dropna()
male_w = brain.loc[brain["Gender"] == "Male", "Weight"].dropna()
print("t-test weight by gender:", stats.ttest_ind(female_w, male_w))
print("Mann–Whitney VIQ by gender:", stats.mannwhitneyu(female_viq, male_viq, alternative="two-sided"))


1-sample t-test VIQ vs 0: Ttest_1sampResult(statistic=30.088099970849328, pvalue=1.3289196468728067e-28)
2-sample t-test VIQ by gender: Ttest_indResult(statistic=-0.7726161723275011, pvalue=0.44452876778583217)
Paired t-test FSIQ vs PIQ: Ttest_relResult(statistic=1.7842019405859857, pvalue=0.08217263818364236)
Wilcoxon signed-rank FSIQ vs PIQ: WilcoxonResult(statistic=274.5, pvalue=0.10659492713506856)
t-test weight by gender: Ttest_indResult(statistic=-4.870950921940696, pvalue=2.227293018362118e-05)
Mann–Whitney VIQ by gender: MannwhitneyuResult(statistic=164.5, pvalue=0.3422886868727315)


## 3. Linear models: gender, IQ type, and iris species

**Questions:**
- Express the male–female VIQ comparison as OLS.
- Recast FSIQ vs PIQ as a long-format categorical regression.
- In iris data: after accounting for petal length, does species still predict sepal width? Do versicolor and virginica differ?


In [4]:
from statsmodels.formula.api import ols

# Gender effect on VIQ (same idea as the 2-sample t-test)
m_gender = ols("VIQ ~ Gender", data=brain).fit()
print(m_gender.summary())


                            OLS Regression Results                            
Dep. Variable:                    VIQ   R-squared:                       0.015
Model:                            OLS   Adj. R-squared:                 -0.010
Method:                 Least Squares   F-statistic:                    0.5969
Date:                Mon, 27 Jul 2026   Prob (F-statistic):              0.445
Time:                        16:34:58   Log-Likelihood:                -182.42
No. Observations:                  40   AIC:                             368.8
Df Residuals:                      38   BIC:                             372.2
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        109.4500      5.308     20.

In [5]:
# Long format for IQ type comparison
long_iq = pd.concat([
    pd.DataFrame({"iq": brain["FSIQ"], "type": "fsiq"}),
    pd.DataFrame({"iq": brain["PIQ"], "type": "piq"}),
])
m_iqtype = ols("iq ~ type", data=long_iq).fit()
print(m_iqtype.params)
print(m_iqtype.pvalues)
print("Compare to unpaired t-test:", stats.ttest_ind(brain["FSIQ"], brain["PIQ"]))


Intercept      113.450
type[T.piq]     -2.425
dtype: float64
Intercept      2.041800e-45
type[T.piq]    6.427725e-01
dtype: float64
Compare to unpaired t-test: Ttest_indResult(statistic=0.465637596380964, pvalue=0.6427725009414841)


In [6]:
iris = pd.read_csv("iris.csv")
m_iris = ols("sepal_width ~ name + petal_length", data=iris).fit()
print(m_iris.summary())

# Contrast: versicolor vs virginica coefficients (ANOVA-style F-test)
# Parameter order: Intercept, name[T.versicolor], name[T.virginica], petal_length
print(m_iris.f_test([0, 1, -1, 0]))


                            OLS Regression Results                            
Dep. Variable:            sepal_width   R-squared:                       0.478
Model:                            OLS   Adj. R-squared:                  0.468
Method:                 Least Squares   F-statistic:                     44.63
Date:                Mon, 27 Jul 2026   Prob (F-statistic):           1.58e-20
Time:                        16:34:58   Log-Likelihood:                -38.185
No. Observations:                 150   AIC:                             84.37
Df Residuals:                     146   BIC:                             96.41
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              2.9813      0

### Optional: VIQ ~ Gender after adjusting for brain size and body size

**Question:** After removing effects of MRI volume, height, and weight, is there still a gender difference in VIQ?


In [7]:
m_adj = ols("VIQ ~ Gender + MRI_Count + Height + Weight", data=brain).fit()
print(m_adj.summary())


                            OLS Regression Results                            
Dep. Variable:                    VIQ   R-squared:                       0.249
Model:                            OLS   Adj. R-squared:                  0.158
Method:                 Least Squares   F-statistic:                     2.733
Date:                Mon, 27 Jul 2026   Prob (F-statistic):             0.0455
Time:                        16:34:58   Log-Likelihood:                -167.03
No. Observations:                  38   AIC:                             344.1
Df Residuals:                      33   BIC:                             352.2
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        169.7719     90.054      1.

## 4. Wages, education, and gender interaction

**Questions:** How do log wages relate to education and age? Does the education–wage slope differ by gender?


In [8]:
import seaborn as sns

wages = pd.read_csv(
    "wages.txt",
    skiprows=27,
    skipfooter=6,
    sep=None,
    header=None,
    engine="python",
)
wages.columns = [
    "EDUCATION", "SOUTH", "SEX", "EXPERIENCE", "UNION",
    "WAGE", "AGE", "RACE", "OCCUPATION", "SECTOR", "MARR",
]
wages["WAGE"] = np.log10(wages["WAGE"])

sns.pairplot(wages, vars=["WAGE", "AGE", "EDUCATION"], kind="reg", hue="SEX")
plt.show()


<ipython-input-1-4e33d2fcc7b6>:18: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [9]:
# Build analysis frame with readable gender labels
wage_df = pd.DataFrame({
    "education": wages["EDUCATION"],
    "gender": np.choose(wages["SEX"].astype(int), ["male", "female"]),
    "wage": wages["WAGE"],
})

# Interaction: does education pay differently for males vs females?
m_int = ols("wage ~ education * gender", data=wage_df).fit()
print(m_int.summary())


                            OLS Regression Results                            
Dep. Variable:                   wage   R-squared:                       0.198
Model:                            OLS   Adj. R-squared:                  0.194
Method:                 Least Squares   F-statistic:                     43.72
Date:                Mon, 27 Jul 2026   Prob (F-statistic):           2.94e-25
Time:                        16:35:01   Log-Likelihood:                 88.503
No. Observations:                 534   AIC:                            -169.0
Df Residuals:                     530   BIC:                            -151.9
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

## Brief conclusions

- Mean VIQ is far from zero; male–female VIQ differences are **not** significant in this small sample.
- Paired FSIQ–PIQ differences are borderline / non-significant depending on the test.
- Weight differs by gender; iris species effects remain after adjusting for petal length.
- Education strongly predicts log wages; the education×gender interaction is typically weak (p ≈ 0.05 region) — interpret cautiously.
